# 01 Simulate LightCurve with lightcurvelynx

- author : Sylvie Dagoret-Campagne
- creation date : 2026-10-29
- kernel : lightcurvelynx_py313
- python dependances : The kernel `lightcurvelynx_py313` requires `emcee`, `corner` et `tqdm`.

## documentation on lightcurvelynx

- top path for lightcurvelynx : https://lightcurvelynx.readthedocs.io/en/latest/#
- example of simulating snIa : https://lightcurvelynx.readthedocs.io/en/latest/notebooks/simple_snia.html
- example of reading opsim : https://lightcurvelynx.readthedocs.io/en/latest/notebooks/opsim_notebook.html


## Imports

In [ ]:
import numpy as np
import pandas as pd
import os

from lightcurvelynx.astro_utils.passbands import PassbandGroup
from lightcurvelynx.astro_utils.snia_utils import DistModFromRedshift, X0FromDistMod
from lightcurvelynx.graph_state import GraphState
from lightcurvelynx.math_nodes.np_random import NumpyRandomFunc
from lightcurvelynx.models.sncosmo_models import SncosmoWrapperModel
from lightcurvelynx.obstable.lsst_obstable import LSSTObsTable
from lightcurvelynx.simulate import compute_single_noise_free_lightcurve, simulate_lightcurves
from lightcurvelynx.survey_info import SurveyInfo
from lightcurvelynx.utils.extrapolate import LinearDecay
from lightcurvelynx.utils.plotting import plot_lightcurves

## Configuration

### opsim file

- define the DP2 + preLSST cadence (2025/04 - 2026/07)

In [ ]:
path_rubin_sim_data_path_name = "RUBIN_SIM_DATA_DIR"
path_rubin_sim_data_path = os.getenv(path_rubin_sim_data_path_name)
cadence_file = "dp2/prelsst_20260607_visits.db"

In [ ]:
full_cadence_path = os.path.join(path_rubin_sim_data_path, cadence_file)

In [ ]:
full_cadence_path

## Additional imports and output directories

In [ ]:
import sqlite3
import pickle
import warnings

import matplotlib.pyplot as plt
import emcee
from astropy.cosmology import FlatLambdaCDM
from tqdm.auto import tqdm

from lightcurvelynx.math_nodes.ra_dec_sampler import ApproximateMOCSampler
from lightcurvelynx.obstable.opsim import OpSim
from lightcurvelynx.utils.extrapolate import ZeroPadding

warnings.filterwarnings("ignore", category=UserWarning)

NB_TAG = "01_SimSNIa"
DATA_DIR = f"data_{NB_TAG}"
FIG_DIR = f"figs_{NB_TAG}"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)


def save_fig(fig, name):
    """Save a figure as both PDF and PNG in FIG_DIR."""
    for ext in ("pdf", "png"):
        fig.savefig(os.path.join(FIG_DIR, f"{NB_TAG}_{name}.{ext}"), dpi=150, bbox_inches="tight")

## Inspect the DP2 + pre-LSST visits database

The schema of the visits file is not assumed: we first list the tables and columns, then pick the matching
`lightcurvelynx` loader.

In [ ]:
with sqlite3.connect(full_cadence_path) as con:
    tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", con)["name"].tolist()
    print("Tables:", tables)
    for tb in tables:
        cols = pd.read_sql(f"PRAGMA table_info({tb})", con)["name"].tolist()
        print(f"\n{tb}: {len(cols)} columns")
        print(cols)

In [ ]:
# Export a slim copy of the visits table (needed columns only), for offline checks
SLIM_COLS = [
    "observationId",
    "exp_midpt_mjd",
    "observationStartMJD",
    "fieldRA",
    "fieldDec",
    "band",
    "filter",
    "visitExposureTime",
    "numExposures",
    "fiveSigmaDepth",
    "rotSkyPos",
    "seeingFwhmEff",
    "seeingFwhmGeom",
    "skyBrightness",
    "sky_bg_min",
    "sky_bg_median",
    "sky_bg_max",
    "zero_point_min",
    "zero_point_median",
    "zero_point_max",
    "zero_point_1s",
    "airmass",
    "img_type",
    "science_program",
    "observation_reason",
    "target_name",
    "bad_flag",
    "simulated",
    "emulated",
    "cloud_extinction",
    "pixel_scale_median",
    "psf_area_median",
    "night",
    "day_obs",
]
with sqlite3.connect(full_cadence_path) as con:
    tb = tables[0]
    avail = pd.read_sql(f"PRAGMA table_info({tb})", con)["name"].tolist()
    use = [c for c in SLIM_COLS if c in avail]
    slim = pd.read_sql("SELECT " + ", ".join(use) + f" FROM {tb}", con)
slim_path = "visits_slim.db"
if os.path.exists(slim_path):
    os.remove(slim_path)
with sqlite3.connect(slim_path) as con:
    slim.to_sql("observations", con, index=False)
print(len(slim), "rows,", len(use), "columns written to", slim_path)

## Load the cadence as an ObsTable

Three schemas are handled (selected from the column names):
- Rubin CCDVisit-like (`expMidptMJD`, `zeroPoint`, ...) -> `LSSTObsTable.from_ccdvisit_table`
- Science-validation visits (`exp_midpt_mjd`, `zero_point_median`, ...) -> `LSSTObsTable.from_sv_visits_table`
- Simulated OpSim (`observationStartMJD`, `fieldRA`, ...) -> `OpSim`

In [ ]:
TABLE_NAME = "observations" if "observations" in tables else tables[0]

with sqlite3.connect(full_cadence_path) as con:
    raw_visits = pd.read_sql(f"SELECT * FROM {TABLE_NAME}", con)
print(f"Loaded {len(raw_visits)} rows from table '{TABLE_NAME}'")

cols = set(raw_visits.columns)

# from Science Validation visits
if {"exp_midpt_mjd", "zero_point_median"} <= cols:
    schema = "sv_visits"
    # Keep only the needed columns (the table has ~270) and valid science visits.
    # The extra 'filter' column must be dropped: it clashes with the band -> filter mapping.
    keep = [
        "exp_midpt_mjd",
        "fieldRA",
        "fieldDec",
        "band",
        "visitExposureTime",
        "fiveSigmaDepth",
        "rotSkyPos",
        "seeingFwhmEff",
        "sky_bg_median",
        "zero_point_median",
        "airmass",
        "img_type",
    ]
    sv = raw_visits[[c for c in keep if c in cols]].copy()
    sv = sv[sv["band"].isin(list("ugrizy"))].copy()
    if "img_type" in sv:
        print("img_type values:", sv["img_type"].value_counts().to_dict())
        if "OBJECT" in set(sv["img_type"]):
            sv = sv[sv["img_type"] == "OBJECT"].copy()
        sv = sv.drop(columns=["img_type"])
    # lightcurvelynx requires finite values: coerce to numeric, drop NaN/inf and non-physical rows
    req = [
        "exp_midpt_mjd",
        "fieldRA",
        "fieldDec",
        "visitExposureTime",
        "fiveSigmaDepth",
        "rotSkyPos",
        "seeingFwhmEff",
        "sky_bg_median",
        "zero_point_median",
    ]
    for c in req + ["airmass"]:
        sv[c] = pd.to_numeric(sv[c], errors="coerce")
    print(
        "non-finite values per column:",
        {c: int((~np.isfinite(sv[c])).sum()) for c in req if (~np.isfinite(sv[c])).any()},
    )
    good = np.isfinite(sv[req]).all(axis=1)
    good &= (sv["sky_bg_median"] > 0) & (sv["seeingFwhmEff"] > 0) & (sv["visitExposureTime"] > 0)
    good &= sv["zero_point_median"].between(20.0, 35.0)
    sv = sv[good].reset_index(drop=True)
    print(f"{len(sv)} valid science visits kept out of {len(raw_visits)}")
    obs_table = LSSTObsTable.from_sv_visits_table(sv)
elif {"expMidptMJD", "zeroPoint"} <= cols:
    schema = "ccdvisit"
    obs_table = LSSTObsTable.from_ccdvisit_table(raw_visits)
elif "observationStartMJD" in cols and not {"exp_midpt_mjd", "zero_point_median"} <= cols:
    schema = "opsim"
    obs_table = OpSim(raw_visits)
else:
    raise ValueError(f"Unknown visits schema, columns: {sorted(cols)}")

print(f"Selected Schema : {schema}")
# print("Schema detected:", schema)
print("Number of visits in ObsTable:", len(obs_table))
obs_table.head()

In [ ]:
tmin, tmax = obs_table.time_bounds()
print(f"Time range: MJD {tmin:.2f} -> {tmax:.2f} ({tmax - tmin:.0f} days)")
print(obs_table["filter"].value_counts())

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(np.asarray(obs_table["time"], dtype=float), bins=100, color="C0")
ax.set_xlabel("MJD")
ax.set_ylabel("Number of visits")
ax.set_title("Cadence: DP2 + pre-LSST visits")
save_fig(fig, "cadence_time")

In [ ]:
moc = obs_table.build_moc()
fig, ax = obs_table.plot_footprint()
save_fig(fig, "footprint")

## SN Ia population model (SALT3 + Tripp)

- redshift uniform in [`Z_MIN`, `Z_MAX`], positions drawn from the survey footprint (MOC), `t0` uniform over the cadence
- `x1 ~ N(0, X1_SIGMA)`, `c ~ N(0, C_SIGMA)`, `M_abs ~ N(M_ABS_MEAN, M_ABS_SIGMA)`
- `x0` from the Tripp relation with fixed (alpha, beta) and flat LCDM
- SALT3 is defined on rest-frame phases [-20, +50] d, so the simulation window is restricted to that range

In [ ]:
SALT_MODEL = "salt3"
H0, OMEGA_M = 70.0, 0.3
ALPHA, BETA = 0.14, 3.1
M_ABS_MEAN, M_ABS_SIGMA = -19.3, 0.1
X1_SIGMA, C_SIGMA = 1.0, 0.1
Z_MIN, Z_MAX = 0.02, 0.30
PHASE_WINDOW = (-20.0, 50.0)
MB0 = 10.635  # sncosmo convention: mB = -2.5 log10(x0) + MB0
N_SIM = 200
SEED = 42

cosmo = FlatLambdaCDM(H0=H0, Om0=OMEGA_M)

passband_group = PassbandGroup.from_preset(preset="LSST", table_dir=os.path.join(DATA_DIR, "passbands"))

radec_sampler = ApproximateMOCSampler(moc, depth=12, node_label="radec")
z_func = NumpyRandomFunc("uniform", low=Z_MIN, high=Z_MAX)
distmod_func = DistModFromRedshift(z_func, H0=H0, Omega_m=OMEGA_M)
x1_func = NumpyRandomFunc("normal", loc=0.0, scale=X1_SIGMA)
c_func = NumpyRandomFunc("normal", loc=0.0, scale=C_SIGMA)
m_abs_func = NumpyRandomFunc("normal", loc=M_ABS_MEAN, scale=M_ABS_SIGMA)
x0_func = X0FromDistMod(distmod=distmod_func, x1=x1_func, c=c_func, alpha=ALPHA, beta=BETA, m_abs=m_abs_func)
t0_func = NumpyRandomFunc("uniform", low=tmin, high=tmax)

source = SncosmoWrapperModel(
    SALT_MODEL,
    t0=t0_func,
    x0=x0_func,
    x1=x1_func,
    c=c_func,
    ra=radec_sampler.ra,
    dec=radec_sampler.dec,
    redshift=z_func,
    node_label="source",
    time_extrapolation=LinearDecay(50.0),
    wave_extrapolation=ZeroPadding(),
)
survey_info = SurveyInfo(obstable=obs_table, passbands=passband_group, survey_name="LSST")

ValueError: Found invalid values in column 'sky_bg_e'

## Simulate the light curves

In [ ]:
lightcurves = simulate_lightcurves(
    source,
    N_SIM,
    survey_info,
    rest_time_window_offset=PHASE_WINDOW,
    rng=np.random.default_rng(SEED),
    drop_empty_rows=True,  # discard SNe that fall outside the footprint or the time range of the visits
)
params_col = lightcurves["params"].copy()
print(f"Simulated {len(lightcurves)} SNe Ia")
lightcurves[["id", "ra", "dec", "nobs", "t0", "z"]].head()

name 'survey_info' is not defined

In [ ]:
def lc_to_df(idx):
    """Return the clean (finite flux, positive error) light curve of simulated SN `idx` as a DataFrame."""
    nested = lightcurves.iloc[idx]["lightcurve"]
    if nested is None:
        return pd.DataFrame(columns=["mjd", "filter", "flux", "fluxerr"])
    df = pd.DataFrame(nested)
    good = np.isfinite(df["flux"]) & np.isfinite(df["fluxerr"]) & (df["fluxerr"] > 0)
    return df[good].reset_index(drop=True)


def summarize(idx, snr_min=5.0):
    df = lc_to_df(idx)
    det = df[df["flux"] / df["fluxerr"] > snr_min]
    return {
        "idx": idx,
        "z": lightcurves.iloc[idx]["z"],
        "nobs": len(df),
        "ndet": len(det),
        "nbands_det": det["filter"].nunique(),
    }


summary = pd.DataFrame([summarize(i) for i in range(len(lightcurves))])
# Well-sampled SNe: enough S/N>5 points in at least two bands
selected = summary[(summary["ndet"] >= 10) & (summary["nbands_det"] >= 2)]
print(f"{len(selected)} / {len(summary)} SNe pass the selection (>=10 detections in >=2 bands)")
summary.describe()

In [ ]:
# Plot a few of the selected light curves together with the noise-free model
for idx in selected["idx"].values[:3]:
    df = lc_to_df(idx)
    params_dict = params_col.iloc[idx]
    noise_free = compute_single_noise_free_lightcurve(
        source,
        GraphState.from_dict(params_dict),
        passband_group,
        rest_frame_phase_min=PHASE_WINDOW[0],
        rest_frame_phase_max=PHASE_WINDOW[1],
        rest_frame_phase_step=0.5,
    )
    plot_lightcurves(
        fluxes=df["flux"].to_numpy(float),
        times=df["mjd"].to_numpy(float),
        fluxerrs=df["fluxerr"].to_numpy(float),
        filters=df["filter"].to_numpy(str),
        underlying_model=noise_free,
        title=f"SN {idx}: z={params_dict['source.redshift']:.3f}, t0={params_dict['source.t0']:.1f}",
    )

## Bayesian inference of the SALT parameters

For each simulated SN, the posterior of $\theta = (t_0, m_B, x_1, c)$ is sampled with `emcee` at the known
redshift (host spec-z):

$$\ln p(\theta | D) = -\tfrac12 \sum_i \frac{(f_i - M_i(\theta))^2}{\sigma_i^2} + \ln p(\theta) + \text{const}$$

- $M_i(\theta)$ is the *same* `lightcurvelynx` model as in the simulation (passbands, extrapolation, units: nJy)
- priors: $t_0 \sim U$(data range), $m_B \sim U(15, 30)$, $x_1 \sim N(0, \sigma_{x_1})$, $c \sim N(0, \sigma_c)$ truncated
- Gaussian likelihood with the simulated flux errors. Since the model and the generative process are identical,
  this is a closed-loop test: the pulls should be $\sim N(0, 1)$ if the sampler is correct.

In [ ]:
N_WALKERS, N_STEPS, N_BURN = 16, 700, 250
PRIOR_LIMITS = {"mB": (15.0, 30.0), "x1": (-4.0, 4.0), "c": (-0.4, 0.6)}
PARAM_NAMES = ["t0", "mB", "x1", "c"]


def theta_to_state_dict(base_params, theta):
    """Replace the SALT parameters of a simulation parameter dictionary by theta = (t0, mB, x1, c)."""
    d = dict(base_params)
    d["source.t0"] = theta[0]
    d["source.x0"] = 10.0 ** (-0.4 * (theta[1] - MB0))
    d["source.x1"] = theta[2]
    d["source.c"] = theta[3]
    return d


def true_theta(params):
    return np.array(
        [
            params["source.t0"],
            -2.5 * np.log10(params["source.x0"]) + MB0,
            params["source.x1"],
            params["source.c"],
        ]
    )


def fit_sn(idx, rng, n_walkers=N_WALKERS, n_steps=N_STEPS, n_burn=N_BURN):
    """Sample the posterior of (t0, mB, x1, c) for simulated SN `idx` with emcee."""
    df = lc_to_df(idx)
    base_params = dict(params_col.iloc[idx])  # redshift and sky position are fixed to their true values
    t = df["mjd"].to_numpy(float)
    filt = df["filter"].to_numpy(str)
    flux = df["flux"].to_numpy(float)
    err = df["fluxerr"].to_numpy(float)
    t_lo, t_hi = t.min(), t.max()

    def log_prob(theta):
        t0, mB, x1, c = theta
        if not (t_lo < t0 < t_hi):
            return -np.inf
        if not (PRIOR_LIMITS["mB"][0] < mB < PRIOR_LIMITS["mB"][1]):
            return -np.inf
        if not (PRIOR_LIMITS["x1"][0] < x1 < PRIOR_LIMITS["x1"][1]):
            return -np.inf
        if not (PRIOR_LIMITS["c"][0] < c < PRIOR_LIMITS["c"][1]):
            return -np.inf
        log_prior = -0.5 * (x1 / X1_SIGMA) ** 2 - 0.5 * (c / C_SIGMA) ** 2
        state = GraphState.from_dict(theta_to_state_dict(base_params, theta))
        model_flux = source.evaluate_bandfluxes(passband_group, t, filt, state)
        return log_prior - 0.5 * np.sum(((flux - model_flux) / err) ** 2)

    truth = true_theta(base_params)
    p0 = truth + rng.normal(size=(n_walkers, 4)) * np.array([2.0, 0.05, 0.2, 0.02])
    sampler = emcee.EnsembleSampler(n_walkers, 4, log_prob)
    sampler.run_mcmc(p0, n_steps, progress=False)
    chain = sampler.get_chain(discard=n_burn, flat=True)
    return {
        "idx": idx,
        "z": base_params["source.redshift"],
        "truth": truth,
        "chain": chain,
        "tau": sampler.get_autocorr_time(tol=0),
        "acceptance": sampler.acceptance_fraction.mean(),
    }

Run the fits on a subset of the selected SNe (each fit takes ~0.5-1 min). Results are saved after every SN so the
loop can be interrupted and restarted.

In [ ]:
N_FIT = 10
fit_file = os.path.join(DATA_DIR, f"{NB_TAG}_posteriors.pkl")
fit_results = pickle.load(open(fit_file, "rb")) if os.path.exists(fit_file) else {}

fit_rng = np.random.default_rng(SEED + 1)
for idx in tqdm(selected["idx"].values[:N_FIT]):
    if idx in fit_results:
        continue
    fit_results[idx] = fit_sn(idx, fit_rng)
    with open(fit_file, "wb") as f:
        pickle.dump(fit_results, f)

print(f"{len(fit_results)} posteriors available")

## Diagnostics: pulls, convergence, Hubble residuals

In [ ]:
rows = []
for idx, r in fit_results.items():
    ch, tr = r["chain"], r["truth"]
    mean, std = ch.mean(axis=0), ch.std(axis=0)
    row = {
        "idx": idx,
        "z": r["z"],
        "acc": r["acceptance"],
        "tau_max": r["tau"].max(),
        "n_eff_min": len(ch) / r["tau"].max(),
    }
    for k, name in enumerate(PARAM_NAMES):
        row[f"{name}_true"], row[f"{name}_mean"], row[f"{name}_std"] = tr[k], mean[k], std[k]
        row[f"{name}_pull"] = (mean[k] - tr[k]) / std[k]
    # Standardization: mu = mB - M + alpha*x1 - beta*c  with M = M_ABS_MEAN
    mu_chain = ch[:, 1] - M_ABS_MEAN + ALPHA * ch[:, 2] - BETA * ch[:, 3]
    row["mu_mean"], row["mu_std"] = mu_chain.mean(), mu_chain.std()
    row["mu_cosmo"] = cosmo.distmod(r["z"]).value
    rows.append(row)
res = pd.DataFrame(rows)
res[["idx", "z", "acc", "tau_max", "n_eff_min"] + [f"{n}_pull" for n in PARAM_NAMES]].round(2)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for ax, name in zip(axes, PARAM_NAMES):
    ax.hist(res[f"{name}_pull"], bins=np.linspace(-4, 4, 17), density=True, alpha=0.7)
    xs = np.linspace(-4, 4, 200)
    ax.plot(xs, np.exp(-0.5 * xs**2) / np.sqrt(2 * np.pi), "k-", lw=1)
    ax.set_title(f"{name}: mean={res[f'{name}_pull'].mean():.2f}, std={res[f'{name}_pull'].std():.2f}")
    ax.set_xlabel("(posterior mean - truth) / posterior std")
fig.tight_layout()
save_fig(fig, "pulls")

# Empirical coverage of the central credible intervals, per parameter (expected ~ level for a calibrated posterior)
for level in (0.68, 0.95):
    lo, hi = (1 - level) / 2, 1 - (1 - level) / 2
    inside = np.array(
        [
            (r["truth"] >= np.quantile(r["chain"], lo, axis=0))
            & (r["truth"] <= np.quantile(r["chain"], hi, axis=0))
            for r in fit_results.values()
        ]
    )
    print(f"{level:.0%} interval coverage:", dict(zip(PARAM_NAMES, inside.mean(axis=0).round(2))))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(6, 6), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
zz = np.linspace(res["z"].min() * 0.9, res["z"].max() * 1.1, 100)
axes[0].plot(zz, cosmo.distmod(zz).value, "k-", lw=1)
axes[0].errorbar(res["z"], res["mu_mean"], yerr=res["mu_std"], fmt="o")
axes[0].set_ylabel("distance modulus")
resid = res["mu_mean"] - res["mu_cosmo"]
axes[1].errorbar(res["z"], resid, yerr=res["mu_std"], fmt="o")
axes[1].axhline(0, color="k", lw=1)
axes[1].set_xlabel("redshift")
axes[1].set_ylabel("residual")
axes[1].set_title(f"scatter = {resid.std():.3f} mag (input intrinsic scatter: {M_ABS_SIGMA} mag)", fontsize=9)
save_fig(fig, "hubble")

In [ ]:
# Posterior of one SN (corner plot)
import corner

idx0 = list(fit_results.keys())[0]
r0 = fit_results[idx0]
fig = corner.corner(
    r0["chain"], labels=PARAM_NAMES, truths=r0["truth"], quantiles=[0.16, 0.5, 0.84], show_titles=True
)
save_fig(fig, f"corner_sn{idx0}")